# Dia 28 — GitHub Actions & CI/CD para Data Science

**Fase 1 — Python Basics (Dia 28 de 28 — encerramento da fase)** · 27/11/2026 · Gutemberg Rapôso

Este é o **último caderno da Fase 1**. Em vez de um tópico isolado, ele *consolida* várias ferramentas estudadas nas últimas semanas — `pytest` (Dia 10), type hints (Dia 13), logging (Dia 16), empacotamento (Dia 18), configuração em YAML (Dia 22) e validação de dados (Dia 27) — dentro de uma ideia maior: **automatizar a verificação de qualidade a cada `push`**.

### Roteiro
1. Introdução ao CI/CD para Data Science
2. Anatomia de um workflow do GitHub Actions (+ validador em Python com PyYAML)
3. Qualidade de código com o módulo `ast` (`CodeQualityChecker`)
4. Padrões de testes automatizados para DS (`DataPipelineValidator` com o dataset iris)
5. Validação de notebooks e `nbconvert` (`NotebookMetadataChecker`)
6. Visualização: painel do pipeline com dados reais dos notebooks da Fase 1
7. Criando o arquivo `.github/workflows/ds_ci.yml`
8. Retrospectiva da Fase 1
9. Atualização do README

> **Dados usados:** apenas datasets reais e públicos (seaborn `iris`, `tips`, `penguins`; scikit-learn `load_wine`) e os próprios arquivos `.ipynb`/`.py` deste repositório.

## 1. O que é CI/CD e por que importa para Data Science

**CI (Integração Contínua)** é a prática de integrar mudanças de código com frequência e, a cada integração, rodar automaticamente uma bateria de verificações: instalação de dependências, *lint*, testes, execução de scripts. **CD (Entrega/Implantação Contínua)** é o passo seguinte: se tudo passou, o artefato (pacote, modelo, relatório, site) é publicado automaticamente.

Em Data Science, o problema clássico é *"funciona na minha máquina"*:

- um notebook que só roda porque a célula 14 foi executada antes da célula 9;
- uma dependência que mudou de versão e alterou um resultado;
- um arquivo de dados que mudou de esquema sem ninguém perceber;
- uma função de pré-processamento alterada que quebra silenciosamente o treino.

O CI transforma essas verificações manuais (e esquecíveis) em **regras automáticas e reprodutíveis**: um ambiente limpo, criado do zero a cada execução, roda tudo de cima para baixo.

### Conceitos básicos do GitHub Actions

| Conceito | O que é | Analogia |
| --- | --- | --- |
| **Workflow** | Arquivo YAML em `.github/workflows/` que descreve a automação | A "receita" completa |
| **Evento (`on`)** | O gatilho: `push`, `pull_request`, `schedule`, `workflow_dispatch` | Quando cozinhar |
| **Job** | Conjunto de passos que roda em uma máquina; jobs rodam em paralelo, salvo `needs:` | Uma estação da cozinha |
| **Step** | Um comando (`run:`) ou uma ação pronta (`uses:`) | Uma instrução da receita |
| **Runner** | A máquina virtual que executa o job (ex.: `ubuntu-latest`) | O fogão |
| **Action** | Componente reutilizável publicado (ex.: `actions/checkout@v4`) | Um utensílio pronto |
| **Artifact** | Arquivo gerado e guardado pelo workflow (ex.: notebook executado) | O prato servido |

### CI para software vs. CI para DS/ML

| Dimensão | Software tradicional | Data Science / ML |
| --- | --- | --- |
| O que muda | Código | Código **+ dados + modelos + ambiente** |
| Testes típicos | Unitários, integração | Unitários **+ esquema de dados + qualidade estatística + métricas mínimas** |
| Artefato | Binário/pacote | Notebook executado, modelo serializado, relatório, figuras |
| Determinismo | Geralmente alto | Depende de *seeds*, versões de bibliotecas e hardware |
| Custo de execução | Segundos/minutos | Pode ser alto (treino) → usar amostras e *smoke tests* |
| Falha típica | Exceção | **Falha silenciosa**: roda, mas o resultado está errado |

Por isso, um bom CI de DS combina três camadas: **qualidade de código** (lint/AST), **testes** (funções e dados) e **reprodutibilidade** (notebooks executados do zero com `nbconvert --execute`).

In [1]:
# Imports and environment report (helps reproduce results in CI logs)
import ast
import json
import platform
import textwrap
from pathlib import Path
from typing import Any

import matplotlib
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import yaml
from sklearn.datasets import load_wine

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 28

versions = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "scikit-learn": sklearn.__version__,
    "pyyaml": yaml.__version__,
    "nbformat": nbformat.__version__,
}
for name, version in versions.items():
    print(f"{name:<13} {version}")

python        3.11.17
pandas        2.3.3
numpy         1.26.4
matplotlib    3.11.2
seaborn       0.13.2
scikit-learn  1.9.1
pyyaml        6.0.3
nbformat      5.11.1


In [2]:
def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from `start` until a directory containing `.git` or `01_python_basics` is found."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() or (candidate / "01_python_basics").is_dir():
            return candidate
    raise FileNotFoundError("Repository root not found from " + str(current))


REPO_ROOT = find_repo_root()
PHASE1_DIR = REPO_ROOT / "01_python_basics"
print("Repository root :", REPO_ROOT.name)
print("Phase 1 folder  :", PHASE1_DIR.relative_to(REPO_ROOT))
print("Existing workflows:", sorted(p.name for p in (REPO_ROOT / ".github" / "workflows").glob("*.yml")))

Repository root : Portfolio
Phase 1 folder  : 01_python_basics
Existing workflows: ['ds_ci.yml', 'offline-demo-checks.yml']


## 2. Anatomia de um workflow do GitHub Actions

Abaixo está um workflow completo e comentado, pensado para um projeto de DS. Repare nos pontos-chave:

- **`on`**: roda em `push`/`pull_request`, mas só quando arquivos relevantes mudam (`paths`).
- **`permissions: contents: read`**: princípio do menor privilégio — o workflow só lê o repositório.
- **`concurrency`**: cancela execuções antigas do mesmo branch, economizando minutos.
- **`strategy.matrix`**: testa a mesma coisa em várias versões do Python.
- **`needs`**: cria dependência entre jobs (só executa notebooks se o lint passar).
- **`actions/upload-artifact`**: guarda o notebook executado para inspeção.

In [3]:
EXAMPLE_WORKFLOW = """
name: DS example pipeline

on:
  push:
    branches: [master]
    paths: ["**.py", "**.ipynb", "requirements*.txt"]
  pull_request:
  workflow_dispatch:

permissions:
  contents: read

concurrency:
  group: ds-example-${{ github.ref }}
  cancel-in-progress: true

jobs:
  lint:
    runs-on: ubuntu-latest
    timeout-minutes: 10
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install linters
        run: python -m pip install ruff
      - name: Lint (syntax errors and undefined names)
        run: ruff check --select E9,F63,F7,F82 .

  test:
    needs: lint
    runs-on: ubuntu-latest
    strategy:
      fail-fast: false
      matrix:
        python-version: ["3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip
      - name: Install dependencies
        run: python -m pip install pandas seaborn scikit-learn pytest
      - name: Run tests
        run: python -m pytest -q

  notebooks:
    needs: test
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Execute notebooks
        run: jupyter nbconvert --to notebook --execute notebooks/*.ipynb --output-dir executed
      - uses: actions/upload-artifact@v4
        with:
          name: executed-notebooks
          path: executed/
"""

parsed = yaml.safe_load(EXAMPLE_WORKFLOW)
print("Top-level keys:", list(parsed.keys()))
print("Jobs:", list(parsed["jobs"].keys()))

Top-level keys: ['name', True, 'permissions', 'concurrency', 'jobs']
Jobs: ['lint', 'test', 'notebooks']


### Pegadinha do YAML: a chave `on` vira `True`

Repare na saída acima: a chave `on` aparece como **`True`**. Isso acontece porque o PyYAML segue a especificação YAML 1.1, na qual `on`, `yes` e `true` são booleanos. O GitHub interpreta corretamente, mas **qualquer ferramenta Python que leia workflows precisa tratar esse caso** — o validador abaixo faz isso explicitamente.

In [4]:
VALID_EVENTS = {"push", "pull_request", "schedule", "workflow_dispatch", "release", "workflow_call"}


def get_triggers(workflow: dict[Any, Any]) -> Any:
    """Return the `on` block, handling PyYAML's YAML 1.1 parsing of `on` as boolean True."""
    return workflow.get("on", workflow.get(True))


def validate_workflow(workflow: dict[Any, Any]) -> list[str]:
    """Validate the structure of a parsed GitHub Actions workflow.

    Returns a list of human-readable problems (empty list means valid).
    """
    problems: list[str] = []
    if not isinstance(workflow, dict):
        return ["Workflow root must be a mapping"]
    if "name" not in workflow:
        problems.append("Missing top-level 'name'")

    triggers = get_triggers(workflow)
    if triggers is None:
        problems.append("Missing 'on' trigger block")
    else:
        events = [triggers] if isinstance(triggers, str) else list(triggers)
        unknown = [e for e in events if e not in VALID_EVENTS]
        if unknown:
            problems.append(f"Unknown trigger events: {unknown}")

    if "permissions" not in workflow:
        problems.append("No 'permissions' block (least privilege recommended)")

    jobs = workflow.get("jobs")
    if not isinstance(jobs, dict) or not jobs:
        problems.append("Missing or empty 'jobs' mapping")
        return problems

    for job_id, job in jobs.items():
        prefix = f"job '{job_id}'"
        if "runs-on" not in job:
            problems.append(f"{prefix}: missing 'runs-on'")
        if "timeout-minutes" not in job:
            problems.append(f"{prefix}: no 'timeout-minutes' (default is 360)")
        needs = job.get("needs", [])
        for dep in [needs] if isinstance(needs, str) else needs:
            if dep not in jobs:
                problems.append(f"{prefix}: needs unknown job '{dep}'")
        steps = job.get("steps")
        if not steps:
            problems.append(f"{prefix}: no steps")
            continue
        for i, step in enumerate(steps, start=1):
            has_uses, has_run = "uses" in step, "run" in step
            if has_uses == has_run:
                problems.append(f"{prefix} step {i}: must have exactly one of 'uses' or 'run'")
            if has_uses and "@" not in step["uses"]:
                problems.append(f"{prefix} step {i}: action '{step['uses']}' is not pinned to a version")
    return problems


issues = validate_workflow(parsed)
print(f"{len(issues)} issue(s) found in EXAMPLE_WORKFLOW:")
for issue in issues:
    print("  -", issue)

2 issue(s) found in EXAMPLE_WORKFLOW:
  - job 'test': no 'timeout-minutes' (default is 360)
  - job 'notebooks': no 'timeout-minutes' (default is 360)


O validador encontrou problemas reais no exemplo: os jobs `test` e `notebooks` não têm `timeout-minutes` (o padrão do GitHub é 6 horas — um notebook travado consumiria minutos à toa). Agora vamos testar o validador com um workflow **propositalmente quebrado** para garantir que ele detecta cada tipo de erro.

In [5]:
BROKEN_WORKFLOW = """
on: [push, deploy_to_mars]
jobs:
  build:
    steps:
      - uses: actions/checkout
      - name: Ambiguous step
        uses: actions/setup-python@v5
        run: echo "both uses and run"
  report:
    runs-on: ubuntu-latest
    needs: [build, ghost_job]
    timeout-minutes: 5
    steps: []
"""

for issue in validate_workflow(yaml.safe_load(BROKEN_WORKFLOW)):
    print("  -", issue)

  - Missing top-level 'name'
  - Unknown trigger events: ['deploy_to_mars']
  - No 'permissions' block (least privilege recommended)
  - job 'build': missing 'runs-on'
  - job 'build': no 'timeout-minutes' (default is 360)
  - job 'build' step 1: action 'actions/checkout' is not pinned to a version
  - job 'build' step 2: must have exactly one of 'uses' or 'run'
  - job 'report': needs unknown job 'ghost_job'
  - job 'report': no steps


In [6]:
# Validate the workflow that already exists in this repository (real file)
existing_path = REPO_ROOT / ".github" / "workflows" / "offline-demo-checks.yml"
existing = yaml.safe_load(existing_path.read_text(encoding="utf-8"))
print("Workflow:", existing["name"])
print("Triggers:", list(get_triggers(existing)))
print("Steps in 'verify':", len(existing["jobs"]["verify"]["steps"]))
print("Issues:", validate_workflow(existing) or "none")

Workflow: Offline demonstration checks
Triggers: ['push', 'pull_request']
Steps in 'verify': 7
Issues: none


## 3. Qualidade de código com o módulo `ast`

Ferramentas como **ruff**, **flake8** e **pylint** funcionam lendo o código como uma **árvore sintática abstrata (AST)** — sem executá-lo. O módulo `ast` da biblioteca padrão permite construir verificações próprias. Isso é útil em DS para regras específicas do time, por exemplo:

- toda função pública tem *docstring*?
- os parâmetros e o retorno têm *type hints* (Dia 13)?
- alguma função ficou complexa demais para ser testada?

A **complexidade ciclomática** (McCabe) conta os caminhos independentes de execução: começa em 1 e soma 1 para cada ponto de decisão (`if`, `for`, `while`, `except`, operadores booleanos, expressões condicionais, `if` em compreensões, `case`). Valores acima de ~10 costumam indicar funções difíceis de testar.

In [7]:
SAMPLE_SOURCE = """
def clean_column_names(df):
    return df.rename(columns=lambda c: c.strip().lower().replace(" ", "_"))


def summarize(df: "pd.DataFrame", group: str) -> "pd.DataFrame":
    "Return mean of numeric columns per group."
    return df.groupby(group).mean(numeric_only=True)


def label_tip(total_bill, tip, smoker, day):
    rate = tip / total_bill if total_bill else 0
    if rate > 0.25 and smoker == "Yes":
        label = "generous smoker"
    elif rate > 0.25:
        label = "generous"
    elif rate < 0.10 or day in ("Sat", "Sun"):
        label = "low or weekend"
    else:
        label = "standard"
    for _ in range(1):
        try:
            label = label.upper()
        except AttributeError:
            pass
    return label
"""
print(SAMPLE_SOURCE.count("def "), "functions in sample source")

3 functions in sample source


In [8]:
class CodeQualityChecker:
    """Static (AST-based) quality checks for Python functions."""

    DECISION_NODES = (ast.If, ast.For, ast.AsyncFor, ast.While, ast.ExceptHandler,
                      ast.IfExp, ast.With, ast.AsyncWith, ast.Assert)

    def __init__(self, max_complexity: int = 5) -> None:
        self.max_complexity = max_complexity

    @classmethod
    def cyclomatic_complexity(cls, node: ast.AST) -> int:
        """Basic McCabe complexity: 1 + number of decision points inside `node`."""
        complexity = 1
        for child in ast.walk(node):
            if isinstance(child, cls.DECISION_NODES) and not isinstance(child, (ast.With, ast.AsyncWith)):
                complexity += 1
            elif isinstance(child, ast.BoolOp):
                complexity += len(child.values) - 1
            elif isinstance(child, ast.comprehension):
                complexity += len(child.ifs)
            elif hasattr(ast, "match_case") and isinstance(child, ast.match_case):
                complexity += 1
        return complexity

    @staticmethod
    def type_hint_coverage(func: ast.FunctionDef | ast.AsyncFunctionDef) -> float:
        """Share of annotatable slots (params except self/cls, plus return) that have hints."""
        params = [a for a in func.args.posonlyargs + func.args.args + func.args.kwonlyargs
                  if a.arg not in ("self", "cls")]
        slots = len(params) + 1
        annotated = sum(a.annotation is not None for a in params) + (func.returns is not None)
        return annotated / slots

    def analyze_source(self, source: str, origin: str = "<string>") -> pd.DataFrame:
        """Return one row per function with quality metrics and a pass/fail verdict."""
        tree = ast.parse(source, filename=origin)
        rows = []
        for node in ast.walk(tree):
            if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                continue
            complexity = self.cyclomatic_complexity(node)
            hints = self.type_hint_coverage(node)
            has_doc = ast.get_docstring(node) is not None
            rows.append({
                "origin": origin,
                "function": node.name,
                "line": node.lineno,
                "n_lines": (node.end_lineno or node.lineno) - node.lineno + 1,
                "has_docstring": has_doc,
                "type_hint_coverage": round(hints, 2),
                "complexity": complexity,
                "passes": has_doc and hints == 1.0 and complexity <= self.max_complexity,
            })
        return pd.DataFrame(rows)

    def analyze_paths(self, paths: list[Path], root: Path) -> pd.DataFrame:
        """Analyze several .py files, labelling rows with paths relative to `root`."""
        frames = [self.analyze_source(p.read_text(encoding="utf-8"), str(p.relative_to(root)))
                  for p in paths]
        frames = [f for f in frames if not f.empty]
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    @staticmethod
    def report(results: pd.DataFrame) -> dict[str, Any]:
        """Aggregate a results table into a short quality report."""
        if results.empty:
            return {"functions": 0}
        return {
            "functions": len(results),
            "docstring_rate": round(results["has_docstring"].mean(), 2),
            "mean_type_hint_coverage": round(results["type_hint_coverage"].mean(), 2),
            "max_complexity": int(results["complexity"].max()),
            "pass_rate": round(results["passes"].mean(), 2),
        }


checker = CodeQualityChecker(max_complexity=5)
sample_results = checker.analyze_source(SAMPLE_SOURCE, "sample")
sample_results

,origin,function,line,n_lines,has_docstring,type_hint_coverage,complexity,passes
0,sample,clean_column_names,2,2,False,0.0,1,False
1,sample,summarize,6,3,True,1.0,1,True
2,sample,label_tip,11,16,False,0.0,9,False


In [9]:
print(json.dumps(CodeQualityChecker.report(sample_results), indent=2))

{
  "functions": 3,
  "docstring_rate": 0.33,
  "mean_type_hint_coverage": 0.33,
  "max_complexity": 9,
  "pass_rate": 0.33
}


**Leitura do resultado:** `clean_column_names` falha por falta de docstring e de type hints; `summarize` passa em tudo; `label_tip` tem complexidade alta (vários `if/elif`, operadores `and`/`or`, um `for`, um `try/except` e uma expressão condicional). Em um CI real, esta tabela poderia virar uma regra: *"falhe o build se alguma função nova tiver complexidade > 10"*.

Agora, aplicando o mesmo verificador ao **código Python real** deste repositório (`projects/` e `tests/`):

In [10]:
py_files = sorted([*REPO_ROOT.glob("projects/**/*.py"), *REPO_ROOT.glob("tests/*.py")])
print("Files analyzed:", [str(p.relative_to(REPO_ROOT)) for p in py_files])
repo_results = checker.analyze_paths(py_files, REPO_ROOT)
print(json.dumps(CodeQualityChecker.report(repo_results), indent=2))
repo_results.sort_values("complexity", ascending=False).head(10)

Files analyzed: ['projects/bike_demand/run.py', 'projects/readability_demo/analyze.py', 'tests/test_bike_demand.py', 'tests/test_readability.py']
{
  "functions": 14,
  "docstring_rate": 0.0,
  "mean_type_hint_coverage": 0.0,
  "max_complexity": 11,
  "pass_rate": 0.0
}


,origin,function,line,n_lines,has_docstring,type_hint_coverage,complexity,passes
0,projects/bike_demand/run.py,load_database,12,31,False,0.0,11,False
5,projects/readability_demo/analyze.py,analyze,22,13,False,0.0,5,False
4,projects/readability_demo/analyze.py,syllables,13,7,False,0.0,4,False
2,projects/bike_demand/run.py,main,49,41,False,0.0,4,False
7,tests/test_bike_demand.py,test_dates_totals_and_sql_agree_with_independe...,15,12,False,0.0,3,False
10,tests/test_readability.py,test_empty_and_punctuation,12,5,False,0.0,2,False
3,projects/readability_demo/analyze.py,words,9,2,False,0.0,1,False
1,projects/bike_demand/run.py,query,45,2,False,0.0,1,False
8,tests/test_bike_demand.py,test_constraints_reject_inconsistent_total,28,4,False,0.0,1,False
6,projects/readability_demo/analyze.py,main,37,6,False,0.0,1,False


In [11]:
# Per-file summary: where would a stricter CI rule bite first?
if not repo_results.empty:
    per_file = (repo_results.groupby("origin")
                .agg(functions=("function", "size"),
                     docstring_rate=("has_docstring", "mean"),
                     hint_coverage=("type_hint_coverage", "mean"),
                     max_complexity=("complexity", "max"))
                .round(2))
    display(per_file)

,functions,docstring_rate,hint_coverage,max_complexity
origin,,,,
projects/bike_demand/run.py,3,0.0,0.0,11
projects/readability_demo/analyze.py,4,0.0,0.0,5
tests/test_bike_demand.py,3,0.0,0.0,3
tests/test_readability.py,4,0.0,0.0,2


> **Observação honesta:** a regra "docstring + 100% de type hints" é deliberadamente rígida para fins didáticos. Funções de teste (`test_*`), por convenção, raramente têm type hints, então uma taxa de aprovação baixa em `tests/` **não** significa código ruim — significa que a regra precisa ser calibrada por contexto (por exemplo, ignorar `test_*`). Essa calibração é exatamente o que se faz ao configurar `ruff`/`pylint` em um projeto real.

## 4. Padrões de testes automatizados para DS

No Dia 10 estudamos `pytest` em profundidade. Aqui o foco é **como esses testes entram no CI**. Três padrões são essenciais:

1. **Fixtures** — preparam dados reutilizáveis (ex.: carregar o iris uma única vez por sessão).
2. **`@pytest.mark.parametrize`** — roda o mesmo teste com várias entradas, gerando um caso por combinação.
3. **`conftest.py`** — arquivo onde ficam fixtures compartilhadas por todos os testes de uma pasta; o pytest o descobre automaticamente.

Primeiro, criamos a classe que será testada: um `DataPipelineValidator` que verifica um DataFrame contra um contrato simples.

In [12]:
class DataPipelineValidator:
    """Validate a DataFrame against a lightweight contract (schema, nulls, ranges, categories)."""

    def __init__(self, expected_dtypes: dict[str, str], ranges: dict[str, tuple[float, float]] | None = None,
                 categories: dict[str, set[str]] | None = None, min_rows: int = 1) -> None:
        self.expected_dtypes = expected_dtypes
        self.ranges = ranges or {}
        self.categories = categories or {}
        self.min_rows = min_rows

    def check_schema(self, df: pd.DataFrame) -> list[str]:
        """Missing columns and dtype-kind mismatches ('f' float, 'i' int, 'O' object...)."""
        errors = [f"missing column '{c}'" for c in self.expected_dtypes if c not in df.columns]
        for col, kind in self.expected_dtypes.items():
            if col in df.columns and df[col].dtype.kind != kind:
                errors.append(f"column '{col}' has kind '{df[col].dtype.kind}', expected '{kind}'")
        return errors

    def check_nulls(self, df: pd.DataFrame) -> list[str]:
        """Report columns of the contract containing missing values."""
        cols = [c for c in self.expected_dtypes if c in df.columns]
        nulls = df[cols].isna().sum()
        return [f"column '{c}' has {n} null(s)" for c, n in nulls.items() if n > 0]

    def check_ranges(self, df: pd.DataFrame) -> list[str]:
        """Report values outside closed [low, high] intervals."""
        errors = []
        for col, (low, high) in self.ranges.items():
            if col in df.columns:
                bad = int((~df[col].between(low, high) & df[col].notna()).sum())
                if bad:
                    errors.append(f"column '{col}' has {bad} value(s) outside [{low}, {high}]")
        return errors

    def check_categories(self, df: pd.DataFrame) -> list[str]:
        """Report unexpected labels in categorical columns."""
        errors = []
        for col, allowed in self.categories.items():
            if col in df.columns:
                unexpected = set(df[col].dropna().astype(str)) - allowed
                if unexpected:
                    errors.append(f"column '{col}' has unexpected values {sorted(unexpected)}")
        return errors

    def validate(self, df: pd.DataFrame) -> dict[str, Any]:
        """Run every check and return a structured report."""
        errors = []
        if len(df) < self.min_rows:
            errors.append(f"only {len(df)} rows, expected >= {self.min_rows}")
        for check in (self.check_schema, self.check_nulls, self.check_ranges, self.check_categories):
            errors.extend(check(df))
        return {"rows": len(df), "valid": not errors, "errors": errors}


iris = sns.load_dataset("iris")
IRIS_CONTRACT = DataPipelineValidator(
    expected_dtypes={"sepal_length": "f", "sepal_width": "f", "petal_length": "f",
                     "petal_width": "f", "species": "O"},
    # Domain-plausible bounds in cm (not derived from this sample's min/max)
    ranges={"sepal_length": (0, 10), "sepal_width": (0, 6), "petal_length": (0, 8), "petal_width": (0, 3)},
    categories={"species": {"setosa", "versicolor", "virginica"}},
    min_rows=100,
)
print(iris.shape)
print(IRIS_CONTRACT.validate(iris))

(150, 5)
{'rows': 150, 'valid': True, 'errors': []}


In [13]:
# Simulate realistic data-quality incidents on a copy of iris
corrupted = iris.copy()
corrupted.loc[3, "sepal_length"] = np.nan          # missing measurement
corrupted.loc[10, "petal_width"] = 25.0            # unit error (mm instead of cm)
corrupted.loc[20, "species"] = "Setosa "           # inconsistent label
corrupted = corrupted.drop(columns="sepal_width")  # upstream schema change

report = IRIS_CONTRACT.validate(corrupted)
print("valid:", report["valid"])
for err in report["errors"]:
    print("  -", err)

valid: False
  - missing column 'sepal_width'
  - column 'sepal_length' has 1 null(s)
  - column 'petal_width' has 1 value(s) outside [0, 3]
  - column 'species' has unexpected values ['Setosa ']


### Como esses testes ficariam em arquivos `pytest`

As células abaixo mostram o **conteúdo** de um `conftest.py` e de um `test_iris_pipeline.py`. Não executamos o `pytest` aqui (o foco é o padrão), mas logo depois emulamos o comportamento de *fixture* e *parametrize* em Python puro para provar que os testes passam.

In [14]:
CONFTEST_PY = textwrap.dedent("""
    # conftest.py — shared fixtures, discovered automatically by pytest
    import pytest
    import seaborn as sns

    @pytest.fixture(scope="session")
    def iris_df():
        "Load the iris dataset once per test session."
        return sns.load_dataset("iris")

    @pytest.fixture
    def iris_copy(iris_df):
        "Function-scoped copy so tests can mutate data safely."
        return iris_df.copy()
""")

TEST_PIPELINE_PY = textwrap.dedent("""
    # test_iris_pipeline.py
    import numpy as np
    import pytest
    from pipeline import IRIS_CONTRACT

    def test_raw_iris_is_valid(iris_df):
        assert IRIS_CONTRACT.validate(iris_df)["valid"]

    @pytest.mark.parametrize("column", ["sepal_length", "petal_width"])
    def test_null_is_detected(iris_copy, column):
        iris_copy.loc[0, column] = np.nan
        assert not IRIS_CONTRACT.validate(iris_copy)["valid"]

    @pytest.mark.parametrize("column,value", [("petal_width", 25.0), ("sepal_length", -1.0)])
    def test_out_of_range_is_detected(iris_copy, column, value):
        iris_copy.loc[0, column] = value
        errors = IRIS_CONTRACT.validate(iris_copy)["errors"]
        assert any(column in e and "outside" in e for e in errors)

    def test_class_balance(iris_df):
        counts = iris_df["species"].value_counts()
        assert counts.min() / counts.max() > 0.8
""")

# Both snippets must at least be syntactically valid Python
for name, src in [("conftest.py", CONFTEST_PY), ("test_iris_pipeline.py", TEST_PIPELINE_PY)]:
    tree = ast.parse(src)
    tests = [n.name for n in tree.body if isinstance(n, ast.FunctionDef)]
    print(f"{name:<24} parses OK -> functions: {tests}")

conftest.py              parses OK -> functions: ['iris_df', 'iris_copy']
test_iris_pipeline.py    parses OK -> functions: ['test_raw_iris_is_valid', 'test_null_is_detected', 'test_out_of_range_is_detected', 'test_class_balance']


In [15]:
# Minimal emulation of fixtures + parametrize to run the same assertions here
def iris_copy_fixture() -> pd.DataFrame:
    """Equivalent of the function-scoped `iris_copy` fixture."""
    return iris.copy()


def test_null_is_detected(df: pd.DataFrame, column: str) -> None:
    df.loc[0, column] = np.nan
    assert not IRIS_CONTRACT.validate(df)["valid"]


def test_out_of_range_is_detected(df: pd.DataFrame, column: str, value: float) -> None:
    df.loc[0, column] = value
    errors = IRIS_CONTRACT.validate(df)["errors"]
    assert any(column in e and "outside" in e for e in errors)


def test_class_balance(df: pd.DataFrame) -> None:
    counts = df["species"].value_counts()
    assert counts.min() / counts.max() > 0.8


cases = [
    ("test_raw_iris_is_valid", lambda: (lambda d: None if IRIS_CONTRACT.validate(d)["valid"] else 1 / 0)(iris_copy_fixture())),
    *[(f"test_null_is_detected[{c}]", lambda c=c: test_null_is_detected(iris_copy_fixture(), c))
      for c in ["sepal_length", "petal_width"]],
    *[(f"test_out_of_range_is_detected[{c}-{v}]", lambda c=c, v=v: test_out_of_range_is_detected(iris_copy_fixture(), c, v))
      for c, v in [("petal_width", 25.0), ("sepal_length", -1.0)]],
    ("test_class_balance", lambda: test_class_balance(iris_copy_fixture())),
]

results = []
for name, run in cases:
    try:
        run()
        results.append((name, "PASSED"))
    except Exception as exc:  # report like pytest would, without stopping the loop
        results.append((name, f"FAILED: {type(exc).__name__}"))

for name, status in results:
    print(f"{status:<8} {name}")
print(f"\n{sum(s == 'PASSED' for _, s in results)} passed, {sum(s != 'PASSED' for _, s in results)} failed")

PASSED   test_raw_iris_is_valid
PASSED   test_null_is_detected[sepal_length]
PASSED   test_null_is_detected[petal_width]
PASSED   test_out_of_range_is_detected[petal_width-25.0]
PASSED   test_out_of_range_is_detected[sepal_length--1.0]
PASSED   test_class_balance

6 passed, 0 failed


### Reutilizando o validador em outros datasets reais

Um bom componente de CI é **genérico**: o mesmo `DataPipelineValidator` funciona para `tips` e `penguins`. No `penguins`, ele detecta corretamente os valores ausentes que existem de fato no dataset original — é exatamente o tipo de alerta que um CI deve emitir antes de um treino.

In [16]:
tips = sns.load_dataset("tips")
penguins = sns.load_dataset("penguins")

tips_contract = DataPipelineValidator(
    expected_dtypes={"total_bill": "f", "tip": "f", "size": "i"},
    ranges={"total_bill": (0, 500), "tip": (0, 100), "size": (1, 20)},
    categories={"time": {"Lunch", "Dinner"}},
)
penguins_contract = DataPipelineValidator(
    expected_dtypes={"bill_length_mm": "f", "flipper_length_mm": "f", "body_mass_g": "f", "species": "O"},
    ranges={"body_mass_g": (1000, 8000)},
    categories={"species": {"Adelie", "Chinstrap", "Gentoo"}},
)
for name, df, contract in [("tips", tips, tips_contract), ("penguins", penguins, penguins_contract)]:
    rep = contract.validate(df)
    print(f"{name:<9} rows={rep['rows']:<4} valid={rep['valid']}")
    for err in rep["errors"]:
        print("           -", err)

tips      rows=244  valid=True
penguins  rows=344  valid=False
           - column 'bill_length_mm' has 2 null(s)
           - column 'flipper_length_mm' has 2 null(s)
           - column 'body_mass_g' has 2 null(s)


In [17]:
# A "model smoke test": a CI-friendly check that training still works and beats a trivial baseline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_wine, y_wine = load_wine(return_X_y=True)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
baseline = DummyClassifier(strategy="most_frequent")

model_scores = cross_val_score(model, X_wine, y_wine, cv=5)
baseline_scores = cross_val_score(baseline, X_wine, y_wine, cv=5)
print(f"wine LogisticRegression 5-fold accuracy: {model_scores.mean():.3f} ± {model_scores.std():.3f}")
print(f"wine majority baseline  5-fold accuracy: {baseline_scores.mean():.3f}")
assert model_scores.mean() > baseline_scores.mean() + 0.2, "Model no longer clearly beats the baseline"
print("Smoke test passed: model clearly beats the trivial baseline.")

wine LogisticRegression 5-fold accuracy: 0.983 ± 0.014
wine majority baseline  5-fold accuracy: 0.399
Smoke test passed: model clearly beats the trivial baseline.


> **Por que um *smoke test* e não uma meta de acurácia rígida?** Um limiar como "acurácia ≥ 0,97" quebraria com pequenas mudanças de versão das bibliotecas. Comparar com uma *baseline* trivial verifica o que importa no CI — **o pipeline ainda aprende algo** — sem transformar uma estimativa de validação cruzada (em um dataset pequeno, com 178 amostras) em um contrato frágil.

## 5. Validação de notebooks e `nbconvert`

Notebooks são arquivos JSON. O comando

```bash
jupyter nbconvert --to notebook --execute caderno.ipynb --output-dir executed/
```

abre o notebook em um **kernel novo**, executa todas as células **de cima para baixo** e salva o resultado. Se qualquer célula lançar uma exceção, o comando termina com código de saída diferente de zero — e o job do GitHub Actions falha. Isso detecta:

- células que dependem de uma ordem de execução fora da sequência;
- variáveis definidas em uma sessão antiga e depois apagadas do código;
- dependências ausentes no ambiente;
- arquivos de dados que não existem no repositório.

Opções úteis: `--ExecutePreprocessor.timeout=600` (limite por célula) e `--ExecutePreprocessor.kernel_name=python3`.

Antes de executar (o que é caro), dá para fazer uma **verificação estática** barata da estrutura — é o que faz o `NotebookMetadataChecker`.

In [18]:
class NotebookMetadataChecker:
    """Static structural checks for .ipynb files (no execution needed)."""

    def __init__(self, min_cells: int = 30) -> None:
        self.min_cells = min_cells

    @staticmethod
    def _execution_in_order(counts: list[int]) -> bool:
        """True when stored execution counts are strictly increasing (top-to-bottom run)."""
        return all(a < b for a, b in zip(counts, counts[1:]))

    @staticmethod
    def validate_schema(nb: nbformat.NotebookNode) -> tuple[bool, bool]:
        """Return (schema_ok, repaired_minor_ids).

        Cell `id` fields only exist from nbformat 4.5. Files saved as 4.4 *with* ids fail strict
        validation but are trivially repairable by bumping the minor version, so they count as a
        warning, not a blocking error.
        """
        try:
            nbformat.validate(nb)
            return True, False
        except nbformat.ValidationError:
            if nb.nbformat_minor < 5 and any("id" in c for c in nb.cells):
                bumped = nbformat.from_dict(dict(nb, nbformat_minor=5))
                try:
                    nbformat.validate(bumped)
                    return True, True
                except nbformat.ValidationError:
                    pass
            return False, False

    def check(self, path: Path) -> dict[str, Any]:
        """Return a flat dict of structural facts and warnings for one notebook."""
        nb = nbformat.read(path, as_version=4)
        schema_ok, minor_ids = self.validate_schema(nb)

        code_cells = [c for c in nb.cells if c.cell_type == "code"]
        md_cells = [c for c in nb.cells if c.cell_type == "markdown"]
        counts = [c.execution_count for c in code_cells if c.execution_count is not None]
        error_outputs = sum(o.get("output_type") == "error" for c in code_cells for o in c.get("outputs", []))
        first_src = nb.cells[0].source.lstrip() if nb.cells else ""

        warnings = []
        if not schema_ok:
            warnings.append("schema")
        if minor_ids:
            warnings.append("ids_in_v4.4")
        if len(nb.cells) < self.min_cells:
            warnings.append("few_cells")
        if not first_src.startswith("#"):
            warnings.append("no_title")
        if error_outputs:
            warnings.append("error_outputs")
        if counts and not self._execution_in_order(counts):
            warnings.append("out_of_order")
        if code_cells and len(counts) < len(code_cells):
            warnings.append("unexecuted_cells")
        if any(not c.source.strip() for c in nb.cells):
            warnings.append("empty_cells")

        return {
            "notebook": path.name,
            "date": path.name[:10],
            "topic": path.stem[11:],
            "n_cells": len(nb.cells),
            "n_code": len(code_cells),
            "n_markdown": len(md_cells),
            "code_lines": sum(len(c.source.splitlines()) for c in code_cells),
            "kernel": nb.metadata.get("kernelspec", {}).get("name", "missing"),
            "schema_ok": schema_ok,
            "error_outputs": error_outputs,
            "executed_share": round(len(counts) / len(code_cells), 2) if code_cells else 0.0,
            "warnings": ",".join(warnings) or "ok",
        }

    def check_folder(self, folder: Path, pattern: str = "20*.ipynb") -> pd.DataFrame:
        """Check every dated notebook in `folder`, sorted chronologically."""
        rows = [self.check(p) for p in sorted(folder.glob(pattern))]
        df = pd.DataFrame(rows)
        df.insert(0, "day", range(1, len(df) + 1))
        return df


nb_checker = NotebookMetadataChecker(min_cells=30)
phase1 = nb_checker.check_folder(PHASE1_DIR)
print(f"{len(phase1)} dated notebooks found in {PHASE1_DIR.name}/")
phase1[["day", "date", "topic", "n_cells", "n_code", "n_markdown", "code_lines", "executed_share", "warnings"]]

28 dated notebooks found in 01_python_basics/


,day,date,topic,n_cells,n_code,n_markdown,code_lines,executed_share,warnings
0,1,2026-09-04,pandas_fundamentals,37,25,12,115,1.0,ids_in_v4.4
1,2,2026-09-18,numpy_fundamentals,37,26,11,290,1.0,ok
2,3,2026-09-19,matplotlib_seaborn,52,25,27,298,1.0,ok
3,4,2026-09-20,python_functions_comprehensions,37,24,13,378,0.0,unexecuted_cells
4,5,2026-10-18,string_manipulation_regex,70,35,35,506,1.0,ok
5,6,2026-10-19,file_io_pandas,42,26,16,379,1.0,ok
6,7,2026-10-20,python_oop_for_ds,43,25,18,617,1.0,ok
7,8,2026-10-21,plotly_interactive_viz,47,32,15,291,1.0,ok
8,9,2026-10-22,datetime_pandas,72,34,38,300,1.0,ok
9,10,2026-10-23,pytest_for_ds,43,22,21,686,1.0,ok


> **Nota:** quando este notebook é executado com `nbconvert --execute --inplace`, o próprio arquivo do Dia 28 já existe na pasta e aparece na tabela. Os números dele refletem a versão salva no disco no momento da execução.

In [19]:
summary = {
    "notebooks": len(phase1),
    "total_cells": int(phase1["n_cells"].sum()),
    "total_code_lines": int(phase1["code_lines"].sum()),
    "all_schema_valid": bool(phase1["schema_ok"].all()),
    "notebooks_with_error_outputs": int((phase1["error_outputs"] > 0).sum()),
    "kernels": phase1["kernel"].value_counts().to_dict(),
}
print(json.dumps(summary, indent=2))

warning_counts = (phase1["warnings"].str.split(",").explode().value_counts().rename("notebooks"))
warning_counts.to_frame()

{
  "notebooks": 28,
  "total_cells": 1481,
  "total_code_lines": 17794,
  "all_schema_valid": true,
  "notebooks_with_error_outputs": 0,
  "kernels": {
    "python3": 28
  }
}


,notebooks
warnings,
ok,25
unexecuted_cells,2
ids_in_v4.4,1


**Como usar isso no CI:** esses fatos viram *gates*. Um bom ponto de partida para este portfólio é:

- **bloquear** (falhar o job) se algum notebook tiver JSON inválido (`schema_ok = False`);
- **avisar** (sem falhar) sobre saídas de erro salvas ou contagens fora de ordem nos notebooks históricos;
- **executar de verdade** com `nbconvert` apenas os notebooks alterados no *push*/PR — re-executar 28 notebooks a cada mudança seria lento e alguns dependem de APIs externas (Dia 25).

Essa estratégia é exatamente a implementada no workflow da Seção 7.

In [20]:
def ci_gate(df: pd.DataFrame) -> tuple[bool, list[str]]:
    """Blocking vs. warning policy used by the workflow's structure job."""
    blocking = df.loc[~df["schema_ok"], "notebook"].tolist()
    warnings = df.loc[df["error_outputs"] > 0, "notebook"].tolist()
    repairable = df.loc[df["warnings"].str.contains("ids_in_v4.4"), "notebook"].tolist()
    messages = [f"::error::{n} is not a valid notebook" for n in blocking]
    messages += [f"::warning::{n} has stored error outputs" for n in warnings]
    messages += [f"::warning::{n} has cell ids but declares nbformat 4.4 (repairable)" for n in repairable]
    return not blocking, messages


passed, messages = ci_gate(phase1)
print("Gate passed:", passed)
print("\n".join(messages) or "(no annotations)")

Gate passed: True
::warning::2026-09-04_pandas_fundamentals.ipynb has cell ids but declares nbformat 4.4 (repairable)


**Achado real:** a validação estrita do `nbformat` reprova o notebook do Dia 1 (`pandas_fundamentals`): ele declara `nbformat_minor = 4`, mas suas células têm o campo `id`, que só existe a partir da versão 4.5. O Jupyter abre o arquivo normalmente, e basta subir a versão menor para que ele seja válido — por isso o `NotebookMetadataChecker` o classifica como **aviso reparável** (`ids_in_v4.4`), e não como erro bloqueante. Sem esse tratamento, o primeiro *push* do workflow já falharia por causa de um detalhe de metadados.

As strings `::error::` e `::warning::` são **comandos de workflow** do GitHub Actions: quando impressas no log, viram anotações visíveis na aba *Checks* do PR. É uma forma simples de tornar o resultado de um script Python legível para quem revisa o código.

## 6. Visualização: painel do pipeline da Fase 1

Agora os dados **reais** coletados pelo `NotebookMetadataChecker` viram um painel: quantas células cada notebook tem, a composição Markdown × código e o progresso acumulado ao longo dos 28 dias.

In [21]:
fig, axes = plt.subplots(2, 1, figsize=(13, 10), gridspec_kw={"height_ratios": [1.2, 1]})

# (a) Stacked bars: cells per notebook (markdown vs code)
ax = axes[0]
labels = [f"D{d} {t[:18]}" for d, t in zip(phase1["day"], phase1["topic"])]
ax.bar(labels, phase1["n_markdown"], color="#4C72B0", label="Markdown")
ax.bar(labels, phase1["n_code"], bottom=phase1["n_markdown"], color="#DD8452", label="Code")
ax.axhline(nb_checker.min_cells, color="grey", ls="--", lw=1, label=f"min. {nb_checker.min_cells} cells")
ax.set_title("Phase 1 — cells per notebook (real .ipynb files)")
ax.set_ylabel("cells")
ax.tick_params(axis="x", rotation=75, labelsize=8)
ax.legend(loc="upper left")

# (b) Line: cumulative progress over the 28 days
ax = axes[1]
cum = phase1.assign(cum_cells=phase1["n_cells"].cumsum(), cum_lines=phase1["code_lines"].cumsum())
ax.plot(cum["day"], cum["cum_cells"], marker="o", color="#4C72B0", label="cumulative cells")
ax.set_xlabel("study day")
ax.set_ylabel("cumulative cells", color="#4C72B0")
ax.set_xticks(range(1, 29))
ax.set_xlim(0.5, 28.5)
ax2 = ax.twinx()
ax2.plot(cum["day"], cum["cum_lines"], marker="s", color="#55A868", label="cumulative code lines")
ax2.set_ylabel("cumulative code lines", color="#55A868")
ax2.grid(False)
ax.set_title(f"Phase 1 progress — {len(phase1)} of 28 days")
handles = ax.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
ax.legend(handles, ["cumulative cells", "cumulative code lines"], loc="upper left")

fig.tight_layout()
figure_path = PHASE1_DIR / "figures" / "phase1_ci_dashboard.png"
figure_path.parent.mkdir(exist_ok=True)
fig.savefig(figure_path, dpi=110, bbox_inches="tight")
plt.show()
print("Saved:", figure_path.relative_to(REPO_ROOT))

Saved: 01_python_basics/figures/phase1_ci_dashboard.png


In [22]:
# Calendar view: notebooks per study block (dates come from file names)
phase1["date_parsed"] = pd.to_datetime(phase1["date"])
phase1["gap_days"] = phase1["date_parsed"].diff().dt.days.fillna(0).astype(int)
blocks = (phase1["gap_days"] > 3).cumsum() + 1
calendar = (phase1.assign(block=blocks)
            .groupby("block")
            .agg(start=("date", "first"), end=("date", "last"), notebooks=("day", "size"),
                 mean_cells=("n_cells", "mean"))
            .round(1))
calendar

,start,end,notebooks,mean_cells
block,,,,
1,2026-09-04,2026-09-04,1,37.0
2,2026-09-18,2026-09-20,3,42.0
3,2026-10-18,2026-10-31,14,55.5
4,2026-11-18,2026-11-27,10,54.1


In [23]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.scatterplot(data=phase1, x="code_lines", y="n_markdown", size="n_cells", hue="day",
                palette="viridis", sizes=(40, 300), ax=ax, legend=False)
for _, row in phase1.nlargest(4, "code_lines").iterrows():
    ax.annotate(f"D{row['day']}", (row["code_lines"], row["n_markdown"]), xytext=(4, 4),
                textcoords="offset points", fontsize=8)
ax.set_title("Code volume vs. explanatory Markdown per notebook (colour = day)")
ax.set_xlabel("lines of code")
ax.set_ylabel("markdown cells")
plt.show()

corr = phase1[["code_lines", "n_markdown"]].corr().iloc[0, 1]
print(f"Pearson r (code lines vs. markdown cells) across {len(phase1)} notebooks: {corr:.2f} (descriptive only)")

Pearson r (code lines vs. markdown cells) across 28 notebooks: 0.01 (descriptive only)


## 7. Criando o arquivo `.github/workflows/ds_ci.yml`

O repositório já possui o workflow `offline-demo-checks.yml`, que testa os projetos de demonstração. O novo workflow **não o substitui**: ele cuida dos notebooks e da qualidade do código. Estrutura:

| Job | O que faz | Bloqueia? |
| --- | --- | --- |
| `lint` | `ruff` com as regras mais críticas (`E9`, `F63`, `F7`, `F82`: erros de sintaxe e nomes indefinidos) em `projects/` e `tests/` | Sim |
| `notebook-structure` | Lê **todos** os `.ipynb` com `nbformat`, valida o esquema JSON e anota saídas de erro e ids em v4.4 como *warnings* | Só se o notebook for inválido |
| `execute-notebooks` | Executa com `nbconvert --execute` **apenas os notebooks alterados** (ou o mais recente da Fase 1, se nenhum mudou) e publica o resultado como *artifact* | Sim |

In [24]:
DS_CI_YAML = """# CI for the Data Science learning notebooks.
# Generated by 01_python_basics/2026-11-27_github_actions_ci_ds.ipynb (Day 28).
# Complements offline-demo-checks.yml, which covers the projects/ demonstrations.
name: DS CI - notebooks and code quality

on:
  push:
    branches: [master]
    paths:
      - "**.ipynb"
      - "**.py"
      - "requirements*.txt"
      - ".github/workflows/ds_ci.yml"
  pull_request:
    paths:
      - "**.ipynb"
      - "**.py"
      - "requirements*.txt"
      - ".github/workflows/ds_ci.yml"
  workflow_dispatch:

permissions:
  contents: read

concurrency:
  group: ds-ci-${{ github.ref }}
  cancel-in-progress: true

jobs:
  lint:
    name: Lint Python sources
    runs-on: ubuntu-latest
    timeout-minutes: 10
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install ruff
        run: python -m pip install ruff
      - name: Syntax errors and undefined names
        run: ruff check --select E9,F63,F7,F82 projects tests

  notebook-structure:
    name: Validate notebook structure
    runs-on: ubuntu-latest
    timeout-minutes: 10
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install nbformat
        run: python -m pip install nbformat
      - name: Validate every notebook (schema blocks, stored errors warn)
        run: |
          python - <<'PY'
          import pathlib, sys
          import nbformat
          failed = 0
          paths = sorted(p for p in pathlib.Path(".").rglob("*.ipynb") if ".ipynb_checkpoints" not in p.parts)
          for path in paths:
              try:
                  nb = nbformat.read(path, as_version=4)
                  try:
                      nbformat.validate(nb)
                  except nbformat.ValidationError:
                      # Cell ids saved under nbformat 4.4: repairable, so warn instead of failing
                      if nb.nbformat_minor < 5 and any("id" in c for c in nb.cells):
                          nbformat.validate(nbformat.from_dict(dict(nb, nbformat_minor=5)))
                          print(f"::warning file={path}::cell ids present but nbformat 4.4 declared")
                      else:
                          raise
              except Exception as exc:
                  failed += 1
                  print(f"::error file={path}::invalid notebook: {str(exc)[:200]}")
                  continue
              errors = sum(o.get("output_type") == "error"
                           for c in nb.cells if c.cell_type == "code" for o in c.get("outputs", []))
              if errors:
                  print(f"::warning file={path}::{errors} stored error output(s)")
          print(f"Checked {len(paths)} notebooks, {failed} invalid")
          sys.exit(1 if failed else 0)
          PY

  execute-notebooks:
    name: Execute changed notebooks
    needs: [lint, notebook-structure]
    runs-on: ubuntu-latest
    timeout-minutes: 30
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - name: Install notebook dependencies
        run: |
          python -m pip install --upgrade pip
          python -m pip install nbconvert nbformat ipykernel pyyaml \\
            pandas numpy matplotlib seaborn scikit-learn scipy
      - name: Select notebooks to execute
        id: select
        env:
          BASE_SHA: ${{ github.event.pull_request.base.sha || github.event.before }}
        run: |
          changed=""
          if [ -n "$BASE_SHA" ] && git cat-file -e "$BASE_SHA" 2>/dev/null; then
            changed=$(git diff --name-only --diff-filter=AM "$BASE_SHA" HEAD -- '*.ipynb' | tr '\\n' ' ')
          fi
          if [ -z "$changed" ]; then
            changed=$(ls 01_python_basics/20*.ipynb | sort | tail -n 1)
          fi
          echo "Notebooks to execute: $changed"
          echo "notebooks=$changed" >> "$GITHUB_OUTPUT"
      - name: Execute with nbconvert
        env:
          MPLBACKEND: Agg
        run: |
          mkdir -p executed
          for nb in ${{ steps.select.outputs.notebooks }}; do
            echo "::group::$nb"
            jupyter nbconvert --to notebook --execute "$nb" \\
              --ExecutePreprocessor.timeout=600 \\
              --ExecutePreprocessor.kernel_name=python3 \\
              --output-dir executed
            echo "::endgroup::"
          done
      - name: Upload executed notebooks
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: executed-notebooks
          path: executed/
          retention-days: 7
"""

ds_ci = yaml.safe_load(DS_CI_YAML)
ci_issues = validate_workflow(ds_ci)
print("Jobs:", list(ds_ci["jobs"]))
print("Validator issues:", ci_issues or "none")
assert not ci_issues, ci_issues

Jobs: ['lint', 'notebook-structure', 'execute-notebooks']
Validator issues: none


Observe que usamos `\\` dentro da string Python para que o arquivo YAML final contenha `\` (continuação de linha no bash) e `'\n'` no comando `tr`. Antes de gravar, o workflow passa pelo **nosso próprio validador** da Seção 2 — o código do notebook "testa" o artefato que ele gera.

In [25]:
workflow_path = REPO_ROOT / ".github" / "workflows" / "ds_ci.yml"
workflow_path.parent.mkdir(parents=True, exist_ok=True)
workflow_path.write_text(DS_CI_YAML, encoding="utf-8")

# Round-trip check: read the file back from disk and validate again
reloaded = yaml.safe_load(workflow_path.read_text(encoding="utf-8"))
assert reloaded == ds_ci
print("Written:", workflow_path.relative_to(REPO_ROOT), f"({workflow_path.stat().st_size} bytes)")
for job_id, job in reloaded["jobs"].items():
    print(f"  {job_id:<20} steps={len(job['steps'])}  needs={job.get('needs', '-')}  timeout={job['timeout-minutes']} min")

Written: .github/workflows/ds_ci.yml (4769 bytes)
  lint                 steps=4  needs=-  timeout=10 min
  notebook-structure   steps=4  needs=-  timeout=10 min
  execute-notebooks    steps=6  needs=['lint', 'notebook-structure']  timeout=30 min


In [26]:
# Sanity check: the embedded Python script in the structure job must be valid Python
structure_run = reloaded["jobs"]["notebook-structure"]["steps"][-1]["run"]
embedded = structure_run.split("<<'PY'\n", 1)[1].rsplit("PY", 1)[0]
ast.parse(textwrap.dedent(embedded))
print("Embedded validation script parses OK")

# Simulate the selection fallback locally: latest dated Phase 1 notebook
latest = sorted(PHASE1_DIR.glob("20*.ipynb"))[-1]
print("Fallback notebook CI would execute:", latest.relative_to(REPO_ROOT))

Embedded validation script parses OK
Fallback notebook CI would execute: 01_python_basics/2026-11-27_github_actions_ci_ds.ipynb


### Explicando cada seção do workflow

- **`on.push.paths` / `on.pull_request.paths`** — o CI só roda quando notebooks, scripts ou requisitos mudam; editar o README não gasta minutos de CI.
- **`permissions: contents: read`** — o workflow não pode escrever no repositório (segurança).
- **`concurrency`** — se eu fizer dois *pushes* seguidos, a execução antiga é cancelada.
- **`lint`** — seleção mínima de regras do `ruff`, que pega erros reais (sintaxe, nomes indefinidos) sem gerar ruído de estilo nos scripts existentes.
- **`notebook-structure`** — script Python embutido via *heredoc*; usa `::error file=...::` e `::warning file=...::` para anotar arquivos.
- **`execute-notebooks`** — `needs` garante que só roda se os jobs anteriores passarem; `fetch-depth: 0` traz o histórico para o `git diff`; `$GITHUB_OUTPUT` passa a lista de notebooks entre *steps*; `MPLBACKEND=Agg` evita depender de interface gráfica.
- **`upload-artifact` com `if: always()`** — mesmo quando a execução falha, o notebook parcialmente executado fica disponível para depuração por 7 dias.

> **Limite conhecido:** notebooks históricos que dependem de bibliotecas extras (por exemplo `plotly`, `pandera`, `httpx`) ou de APIs externas só passarão no job de execução se forem alterados **e** suas dependências forem adicionadas ao passo de instalação. Este workflow foi escrito e validado localmente (estrutura YAML, script embutido e `nbconvert`), mas a primeira execução real no GitHub só acontecerá após o *push*.

## 8. Retrospectiva da Fase 1 — Python Basics (28/28 dias)

### Os 28 tópicos

| Dia | Tópico | Dia | Tópico |
| --- | --- | --- | --- |
| 1 | Fundamentos de pandas | 15 | Gerenciamento de ambientes (venv, pip, conda, uv) |
| 2 | Fundamentos de NumPy | 16 | Logging e depuração |
| 3 | Matplotlib & Seaborn | 17 | Profiling e otimização de desempenho |
| 4 | Funções e *comprehensions* | 18 | Empacotamento e módulos reutilizáveis |
| 5 | Strings e expressões regulares | 19 | SQL com Python (sqlite3, SQLAlchemy) |
| 6 | Leitura/escrita de arquivos com pandas | 20 | CLIs com `argparse` |
| 7 | POO para Data Science | 21 | `dataclasses`, `attrs` e `pydantic` |
| 8 | Visualização interativa com Plotly | 22 | Gerenciamento de configuração |
| 9 | Datas e tempo com pandas | 23 | *Design patterns* para DS |
| 10 | `pytest` para DS | 24 | Classes abstratas e `Protocol` |
| 11 | Decoradores e gerenciadores de contexto | 25 | APIs web com `requests`/`httpx` |
| 12 | `itertools` e `functools` | 26 | Gerenciamento de memória |
| 13 | `typing` e *type hints* | 27 | Validação de dados com Pandera |
| 14 | Concorrência e pipelines paralelos | 28 | **GitHub Actions e CI/CD para DS** |

### O que foi aprendido

1. **Manipulação e visualização de dados (Dias 1–9):** a base — pandas, NumPy, gráficos estáticos e interativos, texto, arquivos e datas.
2. **Python como linguagem de engenharia (Dias 10–14):** testes, decoradores, iteradores preguiçosos, tipagem e paralelismo — escrever código que outra pessoa consegue confiar e manter.
3. **Ferramental profissional (Dias 15–22):** ambientes reprodutíveis, logs, profiling, pacotes, SQL, CLIs, modelos de dados validados e configuração externa.
4. **Arquitetura e robustez (Dias 23–28):** padrões de projeto, interfaces, consumo de APIs, memória, contratos de dados e, hoje, a automação que amarra tudo: **CI/CD**.

### Conexões com minha formação

Como linguista, muitos exemplos da fase envolveram texto: tokenização, regex, frequência de palavras, perfis de dificuldade textual. A ideia de **contrato** (esquemas, validadores, workflows) tem paralelo com a descrição de regras gramaticais: torna explícito o que antes era conhecimento tácito.

### Ponte para a Fase 2 — Análise Exploratória de Dados (EDA)

A Fase 2 (`02_eda/`) usará tudo isso como infraestrutura: cada análise exploratória poderá começar com um **contrato de dados** (Dias 21 e 27), ser organizada em funções testáveis (Dia 10) e, a partir de agora, ser **executada automaticamente no CI** a cada *push*. O foco passa de "como escrever Python" para "como fazer boas perguntas aos dados": distribuições, valores ausentes, *outliers*, relações entre variáveis e comunicação visual dos achados.

In [27]:
# Phase 1 in numbers (computed from the real notebooks, including this one as saved on disk)
phase1_numbers = pd.Series({
    "days completed": f"{len(phase1)} / 28",
    "first notebook": phase1["date"].iloc[0],
    "last notebook": phase1["date"].iloc[-1],
    "total cells": int(phase1["n_cells"].sum()),
    "total code cells": int(phase1["n_code"].sum()),
    "total markdown cells": int(phase1["n_markdown"].sum()),
    "total lines of code": int(phase1["code_lines"].sum()),
    "median cells / notebook": float(phase1["n_cells"].median()),
}, name="Phase 1")
phase1_numbers.to_frame()

,Phase 1
days completed,28 / 28
first notebook,2026-09-04
last notebook,2026-11-27
total cells,1481
total code cells,909
total markdown cells,572
total lines of code,17794
median cells / notebook,53.0


## 9. Atualização do README

Como encerramento, o `README.md` na raiz do repositório foi atualizado (fora deste notebook, como parte do fluxo de publicação do dia) com:

- o *badge* **🔬 Currently Learning: GitHub Actions & CI/CD for Data Science**;
- a tabela de progresso com **Fase 1 = 28/28** e **Fase 2 = 0/28**;
- a lista dos cinco notebooks mais recentes com links relativos;
- o posicionamento *Junior → Senior Data Scientist | Open to Remote Opportunities*;
- links para as pastas temáticas e a seção sobre o background em Linguística + Data Science + NLP aplicado à educação;
- uma seção **"Phase 1 Complete!"**.

A célula abaixo apenas **confere** se o README contém esses elementos — outro exemplo de verificação automatizável em CI.

In [28]:
readme = (REPO_ROOT / "README.md").read_text(encoding="utf-8")
expected_markers = {
    "learning badge": "GitHub Actions & CI/CD for Data Science",
    "phase 1 complete row": "28 / 28",
    "phase 1 complete section": "Phase 1 Complete!",
    "positioning": "Junior → Senior Data Scientist | Open to Remote Opportunities",
    "background section": "Linguistics + Data Science + NLP applied to Education",
    "link to this notebook": "01_python_basics/2026-11-27_github_actions_ci_ds.ipynb",
    "link to phase 2 folder": "(02_eda/)",
}
readme_check = pd.DataFrame(
    [{"marker": k, "present": v in readme} for k, v in expected_markers.items()]
)
readme_check

,marker,present
0,learning badge,True
1,phase 1 complete row,True
2,phase 1 complete section,True
3,positioning,True
4,background section,True
5,link to this notebook,True
6,link to phase 2 folder,True


### Resumo do dia

- **CI/CD** transforma verificações manuais em regras reprodutíveis executadas em um ambiente limpo.
- O **PyYAML** permite validar workflows em Python — cuidado com `on` → `True`.
- O módulo **`ast`** basta para construir checagens de qualidade sob medida (docstrings, *type hints*, complexidade).
- **Fixtures, `parametrize` e `conftest.py`** organizam testes de dados; *smoke tests* contra uma *baseline* evitam limiares frágeis.
- **`nbconvert --execute`** é o teste definitivo de reprodutibilidade de um notebook.
- O workflow `.github/workflows/ds_ci.yml` foi gerado, validado e gravado por este notebook.

**Fase 1 concluída — 28 de 28 dias.** Próxima parada: Fase 2, Análise Exploratória de Dados. 🚀